# Lezione 3 · Come impara una rete

L'errore che risale filo per filo: la backpropagation della terza seduta, rifatta numero per numero.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

## La sera in cui non sei andato

La rete dice 85%, la verità è 0. Seguiamo l'errore che risale, con learning rate η = 0,1.

In [2]:
W1 = np.array([[2.0, 0.0, 1.0], [-3.0, 2.0, 0.0]]); b1 = np.array([-1.0, 0.5])
W2 = np.array([1.5, -2.0]); b2 = -0.5
x = np.array([0.8, 1, 1]); y = 0.0; eta = 0.1

# passaggio in avanti
a = W1 @ x + b1
h = np.maximum(0, a)
z = W2 @ h + b2
p = sigmoide(z)

# primo tempo: l'errore in uscita (sigmoide + cross-entropy)
d = p - y
controlla("segnale d'errore", d, 0.85)

# secondo tempo: la colpa dei fili d'uscita
gW2 = d * h
controlla("colpa h1 -> y", gW2[0], 1.36, 0.02)
controlla("colpa h2 -> y", gW2[1], 0.085, 0.002)

# terzo tempo: l'errore risale ai neuroni nascosti (con i pesi di prima)
dh = d * W2 * (a > 0)
controlla("segnale h1", dh[0], 1.275, 0.02)
controlla("segnale h2", dh[1], -1.7, 0.02)

# quarto tempo: la colpa dei fili nascosti
gW1 = np.outer(dh, x)

# la correzione
W2n, b2n = W2 - eta * gW2, b2 - eta * d
W1n, b1n = W1 - eta * gW1, b1 - eta * dh
controlla("h1 -> y dopo", W2n[0], 1.364, 0.003)
controlla("h2 -> y dopo", W2n[1], -2.009, 0.003)
controlla("energia -> h1 dopo", W1n[0, 0], 1.898, 0.003)
controlla("pioggia -> h1 dopo", W1n[0, 1], -0.128, 0.003)
controlla("energia -> h2 dopo", W1n[1, 0], -2.864, 0.003)
controlla("pioggia -> h2 dopo", W1n[1, 1], 2.17, 0.003)

✓  segnale d'errore: calcolato 0.8455, nel libro 0.85
✓  colpa h1 -> y: calcolato 1.353, nel libro 1.36
✓  colpa h2 -> y: calcolato 0.08455, nel libro 0.085
✓  segnale h1: calcolato 1.268, nel libro 1.275
✓  segnale h2: calcolato -1.691, nel libro -1.7
✓  h1 -> y dopo: calcolato 1.365, nel libro 1.364
✓  h2 -> y dopo: calcolato -2.008, nel libro -2.009
✓  energia -> h1 dopo: calcolato 1.899, nel libro 1.898
✓  pioggia -> h1 dopo: calcolato -0.1268, nel libro -0.128
✓  energia -> h2 dopo: calcolato -2.865, nel libro -2.864
✓  pioggia -> h2 dopo: calcolato 2.169, nel libro 2.17


Ora il passaggio in avanti con i nuovi pesi. Il libro arrotonda il segnale d'errore a 0,85; qui usiamo il valore esatto, 0,8455, quindi i numeri possono differire alla seconda cifra.

In [3]:
hn = np.maximum(0, W1n @ x + b1n)
zn = W2n @ hn + b2n
controlla("h1 nuovo", hn[0], 1.14, 0.01)
controlla("h2 nuovo", hn[1], 0.72, 0.01)
controlla("z nuovo", zn, -0.48, 0.02)
controlla("probabilità nuova", sigmoide(zn), 0.38, 0.01)

✓  h1 nuovo: calcolato 1.138, nel libro 1.14
✓  h2 nuovo: calcolato 0.7155, nel libro 0.72
✓  z nuovo: calcolato -0.4682, nel libro -0.48
✓  probabilità nuova: calcolato 0.385, nel libro 0.38


## Il cantiere dei grandi

70 miliardi di parametri, 15 mila miliardi di token, circa 6 operazioni per parametro e per token.

In [4]:
operazioni = 6 * 70e9 * 15e12
print(f"operazioni: {operazioni:.2e}")
gpu = 4e14                                   # operazioni al secondo, nella pratica
anni = operazioni / gpu / (3600 * 24 * 365)
controlla("anni con una GPU", anni, 500, 10)
giorni = operazioni / (gpu * 10_000) / (3600 * 24)
print(f"giorni con diecimila GPU: {giorni:.1f} (meno di tre settimane)")
assert giorni < 21

parole = 15e12 * 0.75                       # circa tre quarti di parola per token
parole_anno = 250 * 60 * 8 * 365
controlla("milioni di parole lette in un anno", parole_anno / 1e6, 44, 0.5)
controlla("migliaia di anni per leggere tutto", parole / parole_anno / 1e3, 250, 10)

operazioni: 6.30e+24
✓  anni con una GPU: calcolato 499.4, nel libro 500
giorni con diecimila GPU: 18.2 (meno di tre settimane)
✓  milioni di parole lette in un anno: calcolato 43.8, nel libro 44
✓  migliaia di anni per leggere tutto: calcolato 256.8, nel libro 250


## Kata per casa

**Esercizio 1.** Serata (1; 0; 0), verità y = 1, pesi (2,0; −1,5; 0,1), bias −0,5, learning rate 0,5.

In [5]:
w = np.array([2.0, -1.5, 0.1]); b = -0.5
x = np.array([1, 0, 0]); y = 1; eta = 0.5

p = sigmoide(w @ x + b)
controlla("p prima", p, 0.818, 0.001)
d = p - y
controlla("segnale d'errore", d, -0.182, 0.001)
w = w - eta * d * x
b = b - eta * d
controlla("w1 dopo", w[0], 2.091, 0.001)
controlla("bias dopo", b, -0.409, 0.001)
controlla("p dopo", sigmoide(w @ x + b), 0.843, 0.001)

✓  p prima: calcolato 0.8176, nel libro 0.818
✓  segnale d'errore: calcolato -0.1824, nel libro -0.182
✓  w1 dopo: calcolato 2.091, nel libro 2.091
✓  bias dopo: calcolato -0.4088, nel libro -0.409
✓  p dopo: calcolato 0.8432, nel libro 0.843
